In [123]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mlt

In [124]:
data = pd.read_csv('titanic.csv')
data.shape

(891, 12)

In [125]:
#checking for missing data
# data.info()
print(data.head())
data.isnull().sum()

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [126]:
data.pop("Cabin")
data.pop("Name")
data.pop("Ticket")
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          714 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(2)
memory usage: 62.8+ KB
None


In [127]:
print(data.shape)
print(data.head())
print(data.isnull().sum())

(891, 9)
   PassengerId  Survived  Pclass     Sex   Age  SibSp  Parch     Fare Embarked
0            1         0       3    male  22.0      1      0   7.2500        S
1            2         1       1  female  38.0      1      0  71.2833        C
2            3         1       3  female  26.0      0      0   7.9250        S
3            4         1       1  female  35.0      1      0  53.1000        S
4            5         0       3    male  35.0      0      0   8.0500        S
PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
Embarked         2
dtype: int64


In [128]:
# filling missing values
data["Age"]=data["Age"].fillna(data["Age"].mean())

In [129]:
data["Embarked"]=data["Embarked"].fillna(data["Embarked"].mode()[0])

In [130]:
print(data.isnull().sum())
print(data.info())

PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
Embarked       0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
 8   Embarked     891 non-null    object 
dtypes: float64(2), int64(5), object(2)
memory usage: 62.8+ KB
None


In [131]:
data["Pclass"]=data["Pclass"].apply(str)

In [132]:
for col in data.dtypes[data.dtypes=="object"].index :
    for_dummy = data.pop(col)
    data = pd.concat([data,pd.get_dummies(for_dummy, prefix=col)], axis=1)
data.head()

,PassengerId,Survived,Age,SibSp,Parch,Fare,Pclass_1,Pclass_2,Pclass_3,Sex_female,Sex_male,Embarked_C,Embarked_Q,Embarked_S
0,1,0,22.0,1,0,7.2500,0,0,1,0,1,0,0,1
1,2,1,38.0,1,0,71.2833,1,0,0,1,0,1,0,0
2,3,1,26.0,0,0,7.9250,0,0,1,1,0,0,0,1
3,4,1,35.0,1,0,53.1000,1,0,0,1,0,0,0,1
4,5,0,35.0,0,0,8.0500,0,0,1,0,1,0,0,1


In [133]:
labels = data.pop("Survived")

In [134]:
from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(data,labels,test_size=0.25)

In [135]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=10)
rf.fit(X_train, y_train)
rf_predit = rf.predict(X_test)

In [141]:
from sklearn.metrics import accuracy_score
print("accuracy score: ", accuracy_score(y_test, rf_predit))

accuracy score:  0.7937219730941704


In [154]:
from sklearn.ensemble import BaggingClassifier
bagging_model = BaggingClassifier(estimator=RandomForestClassifier(), n_estimators=10,random_state=42)
bagging_model.fit(X_train, y_train)
bagging_predict = bagging_model.predict(X_test)

In [155]:
print("accuracy score: ", accuracy_score(y_test, bagging_predict))

accuracy score:  0.7847533632286996


In [160]:
from sklearn.ensemble import VotingClassifier
voting_classifier = VotingClassifier(estimators=[('rf',RandomForestClassifier())],voting="hard")

voting_classifier.fit(X_train, y_train)
y_predict = voting_classifier.predict(X_test)

In [157]:
print("accuracy score: ", accuracy_score(y_test, y_predict))

accuracy score:  0.820627802690583
